In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries importes successfully")

Libraries importes successfully


In [3]:
script_dir = Path.cwd()
file_path=script_dir.parent/'Datasets'/'survey.csv'
df = pd.read_csv(r"C:\Users\asus\Downloads\forestfires.csv")
print(df.head(5))

   X  Y month  day  FFMC   DMC     DC  ISI  temp  RH  wind  rain  area
0  7  5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0   0.0
1  7  4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0   0.0
2  7  4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0   0.0
3  8  6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2   0.0
4  8  6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0   0.0


In [4]:
print("Dataset Shape:-")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

# Select only numerical columns for statistical analysis
numeric_df = df.select_dtypes(include=[np.number])

print("Basic Statistics (Mean, Median, Std Dev):-")
#Running the analysis on only relevent columns for readability

relevent_df=df[["ISI","temp","RH","wind","area"]]
print(relevent_df.describe(),'\n')

# Covariance:- Shows the directional relationship between variables

print("Covariance Matrix:-")
cov_matrix = relevent_df.cov()
print(cov_matrix,'\n')

# Correlation:- Shows the strength and direction of the linear relationship between variables
print("Correlation Matrix:-")
corr_matrix = relevent_df.corr()
print(corr_matrix)

Dataset Shape:-
Rows: 517, Columns: 13

Basic Statistics (Mean, Median, Std Dev):-
              ISI        temp          RH        wind         area
count  517.000000  517.000000  517.000000  517.000000   517.000000
mean     9.021663   18.889168   44.288201    4.017602    12.847292
std      4.559477    5.806625   16.317469    1.791653    63.655818
min      0.000000    2.200000   15.000000    0.400000     0.000000
25%      6.500000   15.500000   33.000000    2.700000     0.000000
50%      8.400000   19.300000   42.000000    4.000000     0.520000
75%     10.800000   22.800000   53.000000    4.900000     6.570000
max     56.100000   33.300000  100.000000    9.400000  1090.840000 

Covariance Matrix:-
            ISI       temp          RH      wind         area
ISI   20.788832  10.438820   -9.859163  0.872661     2.396689
temp  10.438820  33.716898  -49.969934 -2.362793    36.165676
RH    -9.859163 -49.969934  266.259802  2.029220   -78.441272
wind   0.872661  -2.362793    2.029220  3.21

Starting preprocessing steps

In [5]:
print(relevent_df[relevent_df['area']==0].shape)


(247, 5)


Seeing the IQR we can tell that the area variable is heavily skewed so we apply the logarathmic transform log(1+x) to convert into manageable smallers numbers and better the training

In [6]:
relevent_df['log_area'] = np.log1p(df['area'])

print(relevent_df[['area', 'log_area']].tail(10))

      area  log_area
507   0.00  0.000000
508   0.00  0.000000
509   2.17  1.153732
510   0.43  0.357674
511   0.00  0.000000
512   6.44  2.006871
513  54.29  4.012592
514  11.16  2.498152
515   0.00  0.000000
516   0.00  0.000000


In [7]:
X = relevent_df[["ISI", "temp", "RH", "wind"]]
y = relevent_df["log_area"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (517, 4)
y shape: (517,)


In [17]:
x=relevent_df[['ISI','wind','RH','temp']]
y=relevent_df['log_area']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42) 

model=LinearRegression()

model.fit(X_train,y_train)

y_pred=model.predict(X_test)

def compare(test,pred):
    comparison = pd.DataFrame({
                        "Actual Area": test.to_numpy(),
                        "Predicted Area": pred,
                        "Error": (test - pred)
                                                                                    })
    return comparison

y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(compare(y_test_actual,y_pred_actual).head(5))

     Actual Area  Predicted Area     Error
304         0.00        2.014574 -2.014574
501         0.00        2.042198 -2.042198
441         1.23        2.091167 -0.861167
153         1.46        1.888443 -0.428443
503         1.95        2.147292 -0.197292
